In [ ]:
pip install -U gdal

In [ ]:
# Import the required libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation, ZeroPadding2D, UpSampling2D
from tensorflow.keras.layers import Conv2DTranspose, Conv2D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Reshape, BatchNormalization, LeakyReLU
from tensorflow.keras.models import Sequential, Model

import numpy as np
import matplotlib.pyplot as plt
import cv2
import tifffile
import math

# Define the generator network
def make_generator_model(input_shape):
    model = Sequential()

    model.add(Dense(256 * 16 * 16, activation="relu", input_dim=np.prod(input_shape)))
    model.add(Reshape((16, 16, 256)))

    model.add(Conv2D(512, kernel_size=5, strides=1, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))

    model.add(UpSampling2D())
    model.add(Conv2D(256, kernel_size=5, strides=1, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))

    model.add(UpSampling2D())
    model.add(Conv2D(128, kernel_size=5, strides=1, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))

    model.add(UpSampling2D())
    model.add(Conv2D(64, kernel_size=5, strides=1, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(input_shape[2], kernel_size=5, strides=1, padding="same", activation='tanh'))

    return model


def make_discriminator_model():
    model = models.Sequential()
    
    # Input shape: (batch_size, 256, 256, 5)
    model.add(layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same', input_shape=[256, 256, 5]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (4, 4), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(256, (4, 4), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    
    # Output shape: (batch_size, 16, 16, 512)
    model.add(layers.Conv2D(512, (4, 4), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    
    # Output shape: (batch_size, 1)
    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

generator = make_generator_model((256, 256, 5))
discriminator = make_discriminator_model()

# Define the loss functions
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)


# Define the GAN model as a combination of generator and discriminator models
discriminator.trainable = False
gan_input = keras.Input(shape=(256,256,5))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output, name="gan")

# Compile the models
generator_optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.5)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.5)
discriminator.compile(loss="binary_crossentropy", optimizer=discriminator_optimizer)
gan.compile(loss="binary_crossentropy", optimizer=generator_optimizer)

# Load the .tif images and preprocess them
import os

# Define the root directory where the images are located
root_dir = "./"

# Recursively find all the .tif files in the root directory and its subdirectories
image_paths = []
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith("_UNHEALTHY.tif"):
            image_path = os.path.join(dirpath, filename)
            image_paths.append(image_path)
            
images = []
for path in image_paths:
    # Load the TIFF image
    image = tifffile.imread(path)

    # Display the image shape
    print(image.shape)

    #image = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    # Convert the image to a NumPy array and normalize the pixel values to the range [-1, 1]
    #image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype("float32") / 127.5 - 1.0

    # Resize the image to the desired size using OpenCV
    image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_LINEAR)

    # Append the preprocessed image to the list of images
    images.append(image)

    
train_images = np.array(images)
train_dataset = tf.data.Dataset.from_tensor_slices(train_images)
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(10)

# Define a function to generate images using the trained GAN model
def generate_images(model, noise, epoch):
    # Generate images from noise using the generator model
    generated_images = model.predict(noise)

    # Rescale pixel values to the range [0, 1]
    generated_images = (1/(2*2.25)) * generated_images + 0.5

    # Save the generated images
    for i in range(generated_images.shape[0]):
        tifffile.imwrite(f"generated_images/{epoch}_00002_{i}.tif", generated_images[i])

# Train the GAN model on the preprocessed dataset
epochs = 1000
noise_dim = 256
num_examples_to_generate = 20
seed = tf.random.normal([num_examples_to_generate, noise_dim])

for epoch in range(epochs):
    print(f"Epoch {epoch+1}")
    for real_images in train_dataset:
        # Generate random noise
        
        noise = tf.random.normal([real_images.shape[0], noise_dim])

        # Generate fake images using the generator model
        fake_images = generator.predict(noise)

        # Concatenate real and fake images
        combined_images = tf.concat([real_images, fake_images], axis=0)

        # Create labels for real and fake images
        real_labels = tf.ones((real_images.shape[0], 1))
        fake_labels = tf.zeros((fake_images.shape[0], 1))
        combined_labels = tf.concat([real_labels, fake_labels], axis=0)

        # Train the discriminator model
        discriminator_loss = discriminator.train_on_batch(combined_images, combined_labels)

        # Train the generator model
        noise = tf.random.normal([real_images.shape[0], noise_dim])
        generator_loss = gan.train_on_batch(noise, real_labels)

    # Generate images using the trained GAN model
    if epoch % 100 == 0:
        generate_images(generator, seed, epoch)

# Save the generator model
generator.save("generator_model.h5")